# Matrice transiliente de QdM (proxy sans traceur) + noyau d'entrainement causal

Notebook autonome regroupant les blocs **D1 (construction de b(z,z'))**, **D2 (rang / collapse / noyau parametrique)**
et **E (noyau causal d'entrainement pour Du et le flux de Reynolds)**, avec un **setup commun partage** en tete de
notebook pour ne plus avoir a re-deriver `rho0`, `alt`, `idx_stat`, `dx`, `dy`, etc. dans chaque cellule.

**Corrections apportees par rapport aux versions donnees en chat (elles plantaient) :**
- D1 : la detection de segments (runs verticaux de meme signe de `w`) comparait un tableau 2D (`S[k]`, forme `(ny,nx)`)
  a un scalaire dans un `if` -> `ValueError: truth value of an array is ambiguous`. Remplace par une segmentation
  **entierement vectorisee** (`np.add.at` / `np.minimum.at` / `np.maximum.at`), boucle uniquement sur `t` et `z`
  (quelques dizaines a centaines d'iterations), jamais sur `(y,x)`.
- D1 : chargement en memoire de `Uall`/`Wall` complets (`(nb, n_stat, ny, nx)`) -- plusieurs Go, risque de crash memoire.
  Remplace par un traitement **pas de temps par pas de temps**, avec `del` + `gc.collect()` a chaque iteration.
- D2 : le test de collapse dupliquait le code buggue de D1. Remplace par un **appel a la meme fonction corrigee**
  `build_transilient`, factorisee une fois pour toutes.
- E : referencait des variables non definies dans ce contexte (`Mc_humide`, `Du_humide`, `flux_zt`, `ubar_zt`) ->
  `NameError`. Le bloc est maintenant **entierement autonome** : il recalcule `M_c(z)`, `u_c(z)`, `u_e(z)`, `Du(z)`
  et le flux de Reynolds directement depuis `ua`/`wa`, sans dependre d'un notebook externe.

**Avant de lancer quoi que ce soit : execute la cellule SETUP ci-dessous, en collant ton bloc habituel
(chargement `rho0`, `alt`, `idx_stat`, `dx`, `dy`, `dt_phys`, `n_z`, `n_stat`, `n_x`, `n_y`, `path3d`, `path2d`,
`dim_t`, `dim_z`, `dim_y`, `dim_x`) a la place indiquee. La cellule de controle juste apres te dira immediatement,
et clairement, si quelque chose manque -- plutot qu'un plantage profond dans D1/D2/E.**


## 0. SETUP COMMUN -- a completer une seule fois

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import gc

# ============================================================
#  SETUP -- COLLE ICI TON BLOC HABITUEL (identique a celui
#  utilise dans le notebook principal), qui doit definir :
#
#    path3d(nom)   -> chemin/handle vers un champ 3D (ua, wa, ta, ...)
#    path2d(nom)   -> chemin/handle vers un champ 2D (prw, ...)
#    dim_t, dim_z, dim_y, dim_x  -> noms des dimensions xarray
#    idx_stat      -> indices temporels de l'etat stationnaire
#    n_t, n_z, n_y, n_x, n_stat  -> tailles
#    rho0          -> densite de reference rho0(z), array (n_z,)
#    alt           -> altitudes en metres, array (n_z,)
#    dx, dy        -> pas de grille horizontaux (m)
#    dt_phys       -> pas de temps physique entre sorties 3D (s)
#
#  Colle ce bloc TEL QUEL (celui qui marche deja dans ton notebook
#  principal) -- ne rien recalculer a la main ici.
# ============================================================

# <<<<<<<<<<<<<<<<<<<<<<<<<<<<  COLLER ICI  >>>>>>>>>>>>>>>>>>>>>>>>>>>>


# >>>>>>>>>>>>>>>>>>>>>>>>>>>>  FIN DU COLLAGE  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<

zkm = alt / 1000.0


In [ ]:
# ---- controle : verifie que le setup est complet AVANT d'aller plus loin ----
_required = ['path3d', 'path2d', 'dim_t', 'dim_z', 'dim_y', 'dim_x',
             'idx_stat', 'n_z', 'n_stat', 'n_x', 'n_y', 'rho0', 'alt', 'dx', 'dy', 'dt_phys']
_missing = [v for v in _required if v not in globals()]
if _missing:
    raise NameError(
        "Setup incomplet -- variables manquantes : " + ", ".join(_missing) +
        "\n=> colle ton bloc habituel dans la cellule SETUP ci-dessus, puis relance."
    )
print("Setup OK :")
print(f"  n_z={n_z}  n_stat={n_stat}  n_y={n_y}  n_x={n_x}")
print(f"  dx={dx:.0f} m   dy={dy:.0f} m   dt_phys={dt_phys/3600:.1f} h")
print(f"  alt : {alt.min():.0f} -> {alt.max():.0f} m   ({n_z} niveaux)")


## 1. Fonction commune : segmentation verticale vectorisee

Un "eddy" est proxyfie par un **segment vertical contigu, dans une colonne (y,x) a un instant t donne, ou `w`
garde un signe constant et depasse un seuil**. L'origine/destination du segment (bas/haut selon le signe)
joue le role de `(z', z)` dans la definition de Romps & Kuang (eq. A1).

**Reserve a garder en tete** : ceci est un proxy eulerien, PAS la methode inject-and-decay du papier
(qui necessite des traceurs radioactifs actifs dans la simulation). Les controles de conservation
(D1, cellule suivante) ne tomberont donc pas au zero machine -- c'est attendu.

La fonction est vectorisee sur `(y,x)` (via `np.add.at` / `np.minimum.at` / `np.maximum.at`) et ne boucle
que sur `z` (quelques dizaines d'iterations) pour chaque pas de temps traite.


In [ ]:
def build_transilient(S, W, rho_col, min_run=2):
    """
    Construit Btilde[dest, orig] = flux de masse (kg m^-2 s^-1, non normalise
    par surface/temps) transporte par des segments verticaux coherents.

    S : (nz, ny, nx) signe de w (deja seuille : +1/-1/0), UN SEUL pas de temps
    W : (nz, ny, nx) vitesse verticale, meme pas de temps
    rho_col : (nz,) densite de reference sur la meme grille reduite que S/W
    min_run : nb minimal de niveaux pour qu'un segment compte comme un "eddy"

    Retourne Btilde partiel (a accumuler sur tous les pas de temps par
    l'appelant), forme (nz, nz).
    """
    nz, ny, nx = S.shape
    nonzero = S != 0

    start = np.zeros_like(nonzero)
    start[0] = nonzero[0]
    start[1:] = nonzero[1:] & (S[1:] != S[:-1])

    seg_id = np.cumsum(start, axis=0) - 1
    seg_id = np.where(nonzero, seg_id, -1)

    flux_sum = np.zeros((nz, ny, nx))
    count    = np.zeros((nz, ny, nx), dtype=np.int32)
    zmin     = np.full((nz, ny, nx), nz, dtype=np.int32)
    zmax     = np.full((nz, ny, nx), -1, dtype=np.int32)
    sign_of  = np.zeros((nz, ny, nx))

    yy, xx = np.meshgrid(np.arange(ny), np.arange(nx), indexing='ij')

    for k in range(nz):
        valid = nonzero[k]
        if not valid.any():
            continue
        sid = seg_id[k][valid]
        y_v, x_v = yy[valid], xx[valid]
        val = np.abs(rho_col[k] * W[k][valid])
        np.add.at(flux_sum, (sid, y_v, x_v), val)
        np.add.at(count,    (sid, y_v, x_v), 1)
        np.minimum.at(zmin, (sid, y_v, x_v), k)
        np.maximum.at(zmax, (sid, y_v, x_v), k)
        sign_of[sid, y_v, x_v] = S[k][valid]

    ok = count >= min_run
    Btilde = np.zeros((nz, nz))
    if not ok.any():
        return Btilde

    s_idx  = sign_of[ok]
    zlo    = zmin[ok]
    zhi    = zmax[ok]
    meanfl = flux_sum[ok] / np.maximum(count[ok], 1)

    dest = np.where(s_idx > 0, zhi, zlo)
    orig = np.where(s_idx > 0, zlo, zhi)

    np.add.at(Btilde, (dest, orig), meanfl)
    return Btilde


def accumulate_transilient(ib, idx_time, W_SEUIL=0.3, MIN_RUN=2, verbose=True):
    """
    Boucle sur les pas de temps de idx_time (indices dans idx_stat, PAS
    dans le temps global), lit ua/wa niveau-complet par pas de temps
    (memoire bornee : un seul (nz,ny,nx) par variable a la fois),
    accumule Btilde sur toute la fenetre.

    ib : indices (dans la grille verticale complete) des niveaux retenus
    idx_time : indices (dans idx_stat) des pas de temps a utiliser
    """
    zb = alt[ib]; rb = rho0[ib]; nb = ib.size
    Btilde = np.zeros((nb, nb))
    ds_u = xr.open_dataset(path3d('ua'))
    ds_w = xr.open_dataset(path3d('wa'))
    t_global = idx_stat[idx_time]

    for n, it in enumerate(t_global):
        u = ds_u['ua'].isel({dim_t: it, dim_z: ib}).values.astype(np.float32)
        w = ds_w['wa'].isel({dim_t: it, dim_z: ib}).values.astype(np.float32)
        S = np.sign(w); S[np.abs(w) < W_SEUIL] = 0
        Btilde += build_transilient(S, w, rb, min_run=MIN_RUN)
        del u, w, S
        if n % 20 == 0:
            gc.collect()
            if verbose:
                print(f"  t={n}/{len(t_global)}")
    ds_u.close(); ds_w.close(); gc.collect()
    return Btilde, zb, rb, nb


## 2. D1 -- Construction de b(z,z') sur toute la fenetre stationnaire

Bande d'etude 2-18 km (comme dans le reste du stage). `W_SEUIL` et `MIN_RUN` sont les deux seuls parametres
libres -- regarde le message `[ctrl]` en sortie pour juger s'ils sont raisonnables avant d'interpreter quoi
que ce soit.


In [ ]:
Z_LO, Z_HI = 2000.0, 18000.0
band = (alt >= Z_LO) & (alt <= Z_HI)
ib = np.where(band)[0]

W_SEUIL = 0.3     # m/s, seuil de coherence verticale (cf. reserve methodologique ci-dessus)
MIN_RUN = 2        # nb minimal de niveaux pour qu'un segment compte

Btilde, zb, rb, nb = accumulate_transilient(ib, np.arange(n_stat), W_SEUIL=W_SEUIL, MIN_RUN=MIN_RUN)

dz_b = np.gradient(zb)
norm = n_stat * n_y * n_x * dx * dy * np.outer(dz_b, dz_b)
Btilde_norm = Btilde / np.maximum(norm, 1e-30)

print(f"[ctrl] Btilde >= 0 hors diagonale : "
      f"{(Btilde_norm[~np.eye(nb, dtype=bool)] < -1e-12).sum()} valeurs negatives (attendu : 0, Btilde~ est une somme de |.|)")


In [ ]:
# --- M(z) : flux de masse compensatoire (subsidence), eq. (2) de Romps & Kuang ---
M = np.zeros(nb)
for i in range(nb):
    up = Btilde_norm[:i, i+1:].sum() if i + 1 < nb else 0.0
    down = Btilde_norm[i+1:, :i].sum() if i > 0 else 0.0
    M[i] = up - down

# --- assemblage de b(z,z') complet, eq. (3) (diffusion des petits eddies omise -> residu) ---
b_full = Btilde_norm.copy()
for i in range(nb):
    b_full[i, i] -= Btilde_norm[:, i].sum()      # depart d'eddies partant de z=i
dM = np.gradient(M, zb)
for i in range(nb):
    b_full[i, i] += dM[i]                         # subsidence compensatoire (approx diagonale)

# --- controles de conservation (eq. 5-6 de Romps & Kuang) ---
e_col = np.abs(b_full.sum(axis=0)).max()
e_row = np.abs(b_full.sum(axis=1)).max()
scale = np.abs(b_full).max()
print(f"[ctrl] conservation colonne (source nette a z')  : {e_col:.2e}  (/ max|b| = {e_col/scale:.2%})")
print(f"[ctrl] conservation ligne   (source nette a z)   : {e_row:.2e}  (/ max|b| = {e_row/scale:.2%})")
print("(non-zero attendu : proxy eulerien + terme diffusif des petits eddies omis.")
print(" Si le ratio depasse ~30-50%, resserrer W_SEUIL avant d'interpreter b_full.)")

fig, ax = plt.subplots(figsize=(6, 5.5))
vmax = np.percentile(np.abs(b_full), 98)
im = ax.pcolormesh(zb/1e3, zb/1e3, b_full, cmap='RdBu_r', vmin=-vmax, vmax=vmax, shading='auto')
ax.plot(zb/1e3, zb/1e3, 'k--', lw=.7)
ax.set_xlabel("z' (km, origine)"); ax.set_ylabel('z (km, destination)')
ax.set_title(r"$b(z,z^\prime)$ diagnostiquee (proxy eulerien)")
fig.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()


## 3. D2 -- De b(z,z') a une parametrisation grande echelle

Trois tests dans l'ordre : rang effectif (SVD), stabilite de la forme entre sous-fenetres temporelles
(test de collapse -- reutilise `accumulate_transilient`, plus de code duplique), puis ajustement d'un
noyau parametrique simple si la forme est stable.


In [ ]:
# ---- A. rang effectif (SVD) --------------------------------------------
Uo, So, Vt = np.linalg.svd(b_full)
energie = np.cumsum(So**2) / np.sum(So**2)
r90 = int(np.searchsorted(energie, 0.90)) + 1
print(f"rang pour 90% de l'energie : {r90} / {nb}")
print(f"valeurs singulieres (normalisees) : {np.array2string(So[:8]/So[0], precision=3)}")


In [ ]:
# ---- B. test de collapse : b(t) = A(t) * S(z,z') -------------------------
NWIN = 4
edges = np.linspace(0, n_stat, NWIN + 1).astype(int)
formes, amps = [], []
for w in range(NWIN):
    idx_win = np.arange(edges[w], edges[w + 1])
    Bw, _, _, _ = accumulate_transilient(ib, idx_win, W_SEUIL=W_SEUIL, MIN_RUN=MIN_RUN, verbose=False)
    A_w = np.linalg.norm(Bw)
    formes.append(Bw / (A_w + 1e-30))
    amps.append(A_w)
    print(f"  fenetre {w}: ||B||={A_w:.3e}")

formes = np.array(formes)
ref = np.abs(formes.mean(axis=0))
dispersion = formes.std(axis=0) / np.maximum(ref, ref.max() * 1e-6)
mask_signif = ref > np.percentile(ref, 75)   # ne juger la dispersion que la ou le signal est fort
print(f"\ndispersion relative de la forme normalisee (zones significatives) : "
      f"mediane {np.nanmedian(dispersion[mask_signif]):.2f}")
print("(<0.3 => collapse credible ; >1 => la forme change trop pour figer un noyau unique)")

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(range(NWIN), amps)
ax.set_xlabel('fenetre temporelle'); ax.set_ylabel('||b|| (norme de Frobenius)')
ax.set_title("Amplitude par fenetre (test de collapse)")
plt.tight_layout(); plt.show()


In [ ]:
# ---- C. ajustement parametrique du noyau hors-diagonale -----------------
# forme testee : b_offdiag(z,z\') ~ M0 * sign(z-z\') * exp(-|z-z\'|/L)
from scipy.optimize import curve_fit

off = b_full - np.diag(np.diag(b_full))
def modele(X, M0, L):
    zz, zzp = X
    return M0 * np.sign(zz - zzp) * np.exp(-np.abs(zz - zzp) / max(L, 1e-3))

ZZ, ZZP = np.meshgrid(zb, zb)
mask = ~np.eye(nb, dtype=bool)

kernel_ok = False
try:
    popt, _ = curve_fit(modele, (ZZ[mask], ZZP[mask]), off[mask],
                         p0=[np.abs(off).max(), 3000], maxfev=5000)
    M0, L = popt
    pred = modele((ZZ, ZZP), *popt)
    nse_fit = 1 - np.sum((off[mask] - pred[mask])**2) / np.sum((off[mask] - off[mask].mean())**2)
    print(f"noyau parametrique : M0={M0:.2e} kg/m^4/s, L={L:.0f} m, NSE={nse_fit:+.3f}")
    if nse_fit > 0.3:
        print("=> forme transportable : b(z,z\') = M0 * sign(z-z\') * exp(-|z-z\'|/L)")
        kernel_ok = True
    else:
        print("=> ajustement pauvre : la forme exponentielle simple ne capture pas b(z,z\')")
except RuntimeError:
    print("ajustement du noyau exponentiel a echoue -- la forme n'est pas simple")


## 4. E -- Noyau causal d'entrainement pour Du et le flux de Reynolds

Bloc **entierement autonome** : `M_c(z)`, `u_c(z)`, `u_e(z)`, `Du(z)`, `eps(z)` et le flux de Reynolds sont
recalcules ici depuis `ua`/`wa`, sans dependre du notebook principal.

Rappel du modele : si un updraft conserve approximativement sa quantite de mouvement en montant, diluee par
entrainement, alors `Du(z) = u_c(z) - ubar(z)` obeit a

`Du(z) = - integrale_{z0}^{z} K(z,z') * dubar/dz'(z') dz'`,  `K(z,z') = exp( - integrale_{z'}^{z} eps(z'') dz'' )`

avec `eps(z)` tire de `dz(M_c)/M_c` -- **zero parametre ajuste**.


In [ ]:
W_UP = 0.0   # m/s, seuil definissant un updraft (w > W_UP). A ajuster/tester (ex: 0.5, 1.0).

def diagnostics_bulk_plume(ib, idx_time, w_thresh=0.0):
    """Calcule, moyennes sur idx_time (indices dans idx_stat), les profils
    u_c, u_e, M_c, ubar, et le flux de Reynolds resolu -- tout depuis ua/wa."""
    zb_ = alt[ib]; rb_ = rho0[ib]; nbb = ib.size
    t_global = idx_stat[idx_time]
    nt_ = len(t_global)

    uc_t = np.zeros((nbb, nt_)); ue_t = np.zeros((nbb, nt_))
    Mc_t = np.zeros((nbb, nt_)); ub_t = np.zeros((nbb, nt_))
    Phi_t = np.zeros((nbb, nt_))

    ds_u = xr.open_dataset(path3d('ua'))
    ds_w = xr.open_dataset(path3d('wa'))
    for n, it in enumerate(t_global):
        u = ds_u['ua'].isel({dim_t: it, dim_z: ib}).values.astype(np.float32)
        w = ds_w['wa'].isel({dim_t: it, dim_z: ib}).values.astype(np.float32)
        for k in range(nbb):
            uk, wk = u[k].ravel(), w[k].ravel()
            up = wk > w_thresh
            ub_t[k, n] = uk.mean()
            if up.any() and (~up).any():
                uc_t[k, n] = uk[up].mean()
                ue_t[k, n] = uk[~up].mean()
                sigma_c = up.mean()
                Mc_t[k, n] = rb_[k] * sigma_c * wk[up].mean()
            else:
                uc_t[k, n] = ue_t[k, n] = ub_t[k, n]
            Phi_t[k, n] = rb_[k] * (np.mean(uk * wk) - uk.mean() * wk.mean())
        del u, w
        if n % 20 == 0: gc.collect()
    ds_u.close(); ds_w.close(); gc.collect()

    return dict(zb=zb_, rb=rb_, nb=nbb,
                u_c=uc_t.mean(axis=1), u_e=ue_t.mean(axis=1),
                M_c=Mc_t.mean(axis=1), ubar=ub_t.mean(axis=1),
                Phi=Phi_t.mean(axis=1))

diag = diagnostics_bulk_plume(ib, np.arange(n_stat), w_thresh=W_UP)
zb_e, rb_e, nb_e = diag['zb'], diag['rb'], diag['nb']
Du_diag  = diag['u_c'] - diag['u_e']
Mc_diag  = diag['M_c']
ubar_e   = diag['ubar']
Phi_true = diag['Phi']

print(f"Du_diag : min {Du_diag.min():+.3f}  max {Du_diag.max():+.3f} m/s")
print(f"Mc_diag : min {Mc_diag.min():.2e}  max {Mc_diag.max():.2e} kg m^-2 s^-1")


In [ ]:
# ---- entrainement net eps-delta = dz(Mc)/Mc, partie diluante seulement ----
net = np.gradient(Mc_diag, zb_e) / np.maximum(np.abs(Mc_diag), 1e-30)
eps_diln = np.clip(net, 0, None)   # approx : le detrainement seul n'appauvrit pas ce qui reste

# ---- noyau causal K(z,z') = exp(-int_{z'}^{z} eps dz'') -------------------
cum_eps = np.concatenate([[0.0], np.cumsum(0.5 * (eps_diln[1:] + eps_diln[:-1]) * np.diff(zb_e))])
K = np.zeros((nb_e, nb_e))
for i in range(nb_e):
    for j in range(i + 1):          # causal : z' <= z uniquement
        K[i, j] = np.exp(-(cum_eps[i] - cum_eps[j]))

# ---- prediction de Du par convolution causale, ZERO parametre ajuste -----
dubar_dz = np.gradient(ubar_e, zb_e)
Du_pred = np.array([-np.trapz(K[i, :i+1] * dubar_dz[:i+1], zb_e[:i+1]) for i in range(nb_e)])

nse = 1 - np.sum((Du_diag - Du_pred)**2) / np.sum((Du_diag - Du_diag.mean())**2)
r = np.corrcoef(Du_diag, Du_pred)[0, 1]
print(f"NSE sur Du (zero parametre ajuste) : {nse:+.3f}   correlation r={r:+.3f}")

Phi_pred = Mc_diag * Du_pred
nse_phi = 1 - np.sum((Phi_true - Phi_pred)**2) / np.sum((Phi_true - Phi_true.mean())**2)
r_phi = np.corrcoef(Phi_true, Phi_pred)[0, 1]
print(f"NSE sur le flux de Reynolds directement : {nse_phi:+.3f}   r={r_phi:+.3f}")

with np.errstate(invalid='ignore', divide='ignore'):
    C_local = 1.0 - Du_diag / np.where(np.abs(Du_pred) < 1e-6, np.nan, Du_pred)


In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(19, 5.5))

ax[0].plot(eps_diln * 1e3, zb_e / 1e3, 'darkorange', lw=2)
ax[0].set_xlabel(r'$\varepsilon_{diln}$ ($\times10^{-3}$ m$^{-1}$)')
ax[0].set_title('Entrainement (diagnostique)')

im = ax[1].pcolormesh(zb_e/1e3, zb_e/1e3, K, cmap='viridis', shading='auto')
ax[1].plot(zb_e/1e3, zb_e/1e3, 'r--', lw=.7)
ax[1].set_xlabel("z' (km, origine)"); ax[1].set_ylabel('z (km)')
ax[1].set_title(r'Noyau causal $K(z,z^\prime)$')
fig.colorbar(im, ax=ax[1])

ax[2].plot(Du_diag, zb_e/1e3, 'k-', lw=2.4, label='diagnostique')
ax[2].plot(Du_pred, zb_e/1e3, 'r--', lw=2, label=f'predit (NSE={nse:+.2f})')
ax[2].axvline(0, color='grey', lw=.8); ax[2].set_xlabel('m/s')
ax[2].set_title(r'$\Delta u(z) = u_c - u_e$'); ax[2].legend(fontsize=9)

ax[3].plot(C_local, zb_e/1e3, 'purple', lw=2)
ax[3].axvline(0, color='k', ls=':', lw=1, label='C=0 (dilution pure)')
ax[3].set_xlim(-2, 2); ax[3].set_xlabel(r'$C_{\rm local}(z)$')
ax[3].set_title('Effet de pression residuel'); ax[3].legend(fontsize=8)

for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()
